# 06 · The gross code in Deltakit

**What this notebook is for:** building the gross code with Riverlane's Deltakit, checking it
against stabilizer-qec and IBM, and decoding it both locally (no account) and with Deltakit's
cloud decoders (needs your API token).

**Source:** Deltakit 0.10.2, https://deltakit-docs.riverlane.com ;
code class `deltakit_explorer.codes.BivariateBicycleCode` (implements Bravyi et al., arXiv:2308.07915).
Our helper functions are in `deltakit_gross.py`.

| Part | Needs account? |
|---|---|
| `BivariateBicycleCode`, `css_code_memory_circuit`, `QPU(...).compile_and_add_noise_to_circuit` | No |
| Cloud circuit generation, `BPOSDecoder`, `ACDecoder` (qLDPC decoders) | Yes: `DELTAKIT_TOKEN` |

## 0. Your API token

Get a token from https://deltakit.riverlane.com (sign in → token). Paste it below **between the quotes**
and run the cell. Leave it empty to skip the cloud sections. Don't share the notebook with the token in it.

In [ ]:
DELTAKIT_TOKEN = ""   # <-- paste your token here, e.g. "abc123..."

import os
CLOUD = bool(DELTAKIT_TOKEN.strip())
if CLOUD:
    from deltakit.explorer import Client
    Client.set_token(DELTAKIT_TOKEN.strip())          # saves it for this machine and validates it
    cloud = Client.get_instance()
    print("Deltakit cloud client ready")
else:
    cloud = None
    print("No token: cloud sections will be skipped")

Deltakit cloud client ready


In [2]:
# Setup: make the gross_check folder importable (the .py files live one level up).
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "independence"))
print("gross_check folder:", ROOT)

gross_check folder: C:\Users\kanis\OneDrive\Desktop\my_projects\gross_check\gross_check


## 1. Build the code

Deltakit's arguments: torus `param_l` × `param_m`, `m_A_powers=[a, b, c]` meaning A = x^a + y^b + y^c,
`m_B_powers=[d, e, f]` meaning B = y^d + x^e + x^f. The gross code: 12 × 6, [3, 1, 2], [3, 1, 2].

In [3]:
import numpy as np, stabilizer_qec as sq
from deltakit_gross import deltakit_gross_code, GROSS
print(GROSS)
dk = deltakit_gross_code()
print("n =", dk.n, " k =", dk.k)
HX, HZ = np.array(dk.m_Hx), np.array(dk.m_Hz)
sHX, sHZ = sq.BivariateBicycleCode("gross").check_matrices()
print("same H_X as stabilizer-qec:", np.array_equal(HX, sHX), "| same H_Z:", np.array_equal(HZ, sHZ))

{'param_l': 12, 'param_m': 6, 'm_A_powers': [3, 1, 2], 'm_B_powers': [3, 1, 2]}
n = 144  k = 12
same H_X as stabilizer-qec: True | same H_Z: True


## 2. The memory circuit and its noise

`css_code_memory_circuit` writes a noiseless Z memory; `QPU(...).compile_and_add_noise_to_circuit`
adds a noise model. Deltakit's `ToyNoise(p)` is lighter than IBM's model (idle noise p/10, no reset
errors), so we also add IBM's uniform model ourselves (`add_uniform_circuit_noise`).

Detectors: like IBM's, Deltakit's Z memory has X- and Z-check detectors. IBM's filter mis-handles
Deltakit's layout (it drops the 72 final-readout detectors), so `z_detectors_only` classifies them
from the circuit; the cell checks no single fault goes undetected.

In [4]:
from deltakit_gross import (deltakit_gross_memory_z, deltakit_gross_memory_z_noiseless,
                            add_uniform_circuit_noise, z_detectors_only, undetectable_single_faults)
P = 0.004
toy = z_detectors_only(deltakit_gross_memory_z(P, 12, noise="toy"))
uni = z_detectors_only(add_uniform_circuit_noise(deltakit_gross_memory_z_noiseless(12), P))
for name, c in [("ToyNoise", toy), ("IBM uniform", uni)]:
    print(f"{name:12s} qubits {c.num_qubits}, detectors {c.num_detectors}, observables {c.num_observables}, "
          f"undetectable single faults {undetectable_single_faults(c)}")

ToyNoise     qubits 288, detectors 936, observables 12, undetectable single faults 0
IBM uniform  qubits 288, detectors 936, observables 12, undetectable single faults 0


## 3. Decode locally (no account): BP-OSD with IBM settings

In [5]:
import time
SHOTS = 500
for name, c in [("Deltakit + ToyNoise", toy), ("Deltakit + IBM uniform", uni)]:
    t = time.time()
    dec = sq.BpOsd(c.detector_error_model(decompose_errors=False), max_iter=80, osd_method="osd_cs", osd_order=10)
    d, o = c.compile_detector_sampler(seed=1).sample(SHOTS, separate_observables=True)
    f = int((np.asarray(dec.decode_batch(d), bool) != o).any(axis=1).sum())
    print(f"{name:24s} p={P}: {f}/{SHOTS} = {f/SHOTS:.4f}  ({time.time()-t:.0f}s)")
print("IBM circuit, same decoder, p=0.004 (results_gross.csv): 0.0267")

Deltakit + ToyNoise      p=0.004: 4/500 = 0.0080  (10s)
Deltakit + IBM uniform   p=0.004: 38/500 = 0.0760  (21s)
IBM circuit, same decoder, p=0.004 (results_gross.csv): 0.0267


## 4. Decode with Deltakit's cloud decoders (needs token)

`BPOSDecoder` and `ACDecoder` (Ambiguity Clustering, Riverlane's) are the ones for qLDPC codes.
They take the circuit and a `client`; `decode_batch_to_logical_flip` returns the predicted logical flips.

In [6]:
if CLOUD:
    from deltakit.decode import BPOSDecoder, ACDecoder
    SHOTS = 200
    d, o = uni.compile_detector_sampler(seed=3).sample(SHOTS, separate_observables=True)
    for name, dec in [("BPOSDecoder", BPOSDecoder(circuit=str(uni), parameters={}, client=cloud)),
                      ("ACDecoder", ACDecoder(circuit=str(uni), parameters={"bp_rounds": 200}, client=cloud))]:
        try:
            pred = np.asarray(dec.decode_batch_to_logical_flip(d), bool)
            f = int((pred != o).any(axis=1).sum())
            print(f"{name}: {f}/{SHOTS} = {f/SHOTS:.4f}")
        except Exception as e:
            print(name, "failed:", repr(e)[:300])
else:
    print("skipped (no token)")

BPOSDecoder failed: ValueError('Circuit must have at least one observable. Please make sure your circuit has observables or provide `num_observables` when instantiating an `BPOSDecoder`.')
ACDecoder failed: ValueError('Circuit must have at least one observable. Please make sure your circuit has observables or provide `num_observables` when instantiating an `ACDecoder`.')


## 5. Deltakit's own cloud circuit (needs token)

With a client, `css_code_memory_circuit(..., client=cloud)` asks Riverlane's server for its compiled
circuit for the BB code (it may use a better schedule than the local one).

In [7]:
if CLOUD:
    from deltakit_circuit.gates import PauliBasis
    from deltakit_explorer.codes import css_code_memory_circuit
    try:
        cc = css_code_memory_circuit(dk, num_rounds=12, logical_basis=PauliBasis.Z, client=cloud)
        s = cc.as_stim_circuit()
        print("cloud circuit: qubits", s.num_qubits, "detectors", s.num_detectors)
    except Exception as e:
        print("cloud circuit generation failed:", repr(e)[:300])
else:
    print("skipped (no token)")

cloud circuit: qubits 288 detectors 1728
